# Baseline Colab — MOT15 + MOT16

Этап 1: генерация `.npy` (det + ReID) и прогон оригинального DeepSORT на **6 sequences**.  
Оба набора генерируются одинаково через `generate_detections.py` → `MOT15_train/` и `MOT16_train/`.

In [ ]:
DRIVE_RESOURCES = "/content/drive/MyDrive/DeepSORT/resources"

from google.colab import drive
drive.mount("/content/drive")

import os
assert os.path.isdir(DRIVE_RESOURCES), f"Папка не найдена: {DRIVE_RESOURCES}"

# Клонируем код (HTTPS, без SSH)
if not os.path.isdir("/content/DeepSORT_Project_CV"):
    !git clone https://github.com/Valeriia-Reznik-Dev/DeepSORT_Project_CV.git /content/DeepSORT_Project_CV

%cd /content/DeepSORT_Project_CV
!git pull

# Подключаем ваши данные с Drive
if os.path.islink("resources"):
    os.remove("resources")
elif os.path.isdir("resources"):
    !rm -rf resources
!ln -s "{DRIVE_RESOURCES}" resources

print("resources ->", os.path.realpath("resources"))

In [ ]:
# Colab Python 3.11+: не pin TF 2.13 (не ставится). Используем TF из Colab.
!pip install -q "numpy<2.0.0" opencv-python scipy pyyaml

import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPU", tf.config.list_physical_devices("GPU"))


In [ ]:
import os

MOT15_SEQUENCES = ["TUD-Campus", "TUD-Stadtmitte", "KITTI-17", "PETS09-S2L1"]
MOT16_SEQUENCES = ["MOT16-09", "MOT16-11"]

paths = {
    "model": "resources/networks/mars-small128.pb",
    "mot15_dir": "resources/detections/MOT15/train",
    "mot16_dir": "resources/detections/MOT16/train",
    "mot15_npy_dir": "resources/detections/MOT15_train",
    "mot16_npy_dir": "resources/detections/MOT16_train",
}

assert os.path.isfile(paths["model"]), f"Missing: {paths['model']}"

for seq in MOT15_SEQUENCES:
    assert os.path.isdir(os.path.join(paths["mot15_dir"], seq)), f"Missing MOT15 video: {seq}"
for seq in MOT16_SEQUENCES:
    assert os.path.isdir(os.path.join(paths["mot16_dir"], seq)), f"Missing MOT16 video: {seq}"

print("OK: model + MOT15/MOT16 videos on Drive")

In [ ]:
MODEL = "resources/networks/mars-small128.pb"

# MOT15 + MOT16: один pipeline, generate_detections.py
!python tools/generate_detections.py \
    --model={MODEL} \
    --mot_dir=resources/detections/MOT15/train \
    --output_dir=resources/detections/MOT15_train

!python tools/generate_detections.py \
    --model={MODEL} \
    --mot_dir=resources/detections/MOT16/train \
    --output_dir=resources/detections/MOT16_train

In [ ]:
import numpy as np

ALL_SEQUENCES = [
    (paths["mot15_npy_dir"], MOT15_SEQUENCES),
    (paths["mot16_npy_dir"], MOT16_SEQUENCES),
]

for npy_dir, seqs in ALL_SEQUENCES:
    print(f"--- {npy_dir}")
    for seq in seqs:
        path = os.path.join(npy_dir, f"{seq}.npy")
        assert os.path.isfile(path), f"Missing: {path}"
        print(seq, np.load(path).shape)

print("OK: all 6 .npy ready")

In [ ]:
!python scripts/run_baseline.py


In [ ]:
# Этап 2: TrackEval HOTA на baseline results
!pip install -q "trackeval==1.3.0"

!python scripts/run_eval.py

## Этап 3 — Detector F1 (YOLO + NanoDet + MMDet)

Сравнение детекций с GT: Precision / Recall / F1 для **всех трёх** детекторов.  
Сначала smoke test (`--max-frames 30`), потом полный прогон без флага.

In [ ]:

!python scripts/setup_detectors_colab.py
# веса NanoDet + RTMDet (если ещё нет в resources/models/)
!python scripts/download_detector_models.py

# smoke test — все 3 детектора, GPU автоматически
!python scripts/run_detector_eval.py --detector yolo nanodet mmdet --max-frames 30

# полный прогон
!python scripts/run_detector_eval.py --detector yolo nanodet mmdet

## Этап 4 — ReID clustering metrics (OSNet + ResNet50-IBN + fast-reid)

Standalone-оценка на **GT pedestrian crops**: Silhouette, Calinski–Harabasz, Fowlkes–Mallows.
Сначала smoke test (`--max-frames 30 --max-samples 500`), потом полный прогон.

In [ ]:
!git pull
!python scripts/setup_reid_colab.py   # also repairs mmcv/torch if MMDet was installed
!python scripts/download_reid_models.py

# если mmdet всё ещё падает с mmcv _ext undefined symbol:
# !python scripts/setup_detectors_colab.py --repair-mmcv-only

# smoke test — все 3 ReID модели
!python scripts/run_reid_eval.py --model osnet resnet50_ibn fastreid --max-frames 30 --max-samples 500

# полный прогон
!python scripts/run_reid_eval.py --model osnet resnet50_ibn fastreid

In [ ]:
!zip -r baseline_outputs.zip \
    resources/detections/MOT15_train/ \
    resources/detections/MOT16_train/ \
    results/baseline/original/

from google.colab import files
files.download("baseline_outputs.zip")
